# CosyVoice 2 — Indian English TTS

**Apache 2.0 license** — enterprise/commercial use OK.

Run all cells top to bottom. Tested on RunPod RTX 4090 and Colab A100.

| Step | What | Time |
|------|------|------|
| 1 | Install everything | ~10 min |
| 2 | Download model + data | ~10 min |
| 3 | Load model | ~1 min |
| 4 | Browse & pick reference voices | ~5 min |
| 5 | Generate podcast | ~10 min |
| 6 | Listen & save | ~1 min |

## Step 1: Install Everything

In [ ]:
import os, sys, subprocess

# Detect environment
IS_COLAB = os.path.exists('/content') and 'COLAB_RELEASE_TAG' in os.environ
BASE = '/content' if IS_COLAB else '/workspace'
print(f"Environment: {'Colab' if IS_COLAB else 'RunPod/Other'}")
print(f"Base: {BASE}")

# GPU check
import torch
assert torch.cuda.is_available(), "No GPU!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# System deps
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y sox libsox-dev ffmpeg > /dev/null 2>&1

# Clone CosyVoice
os.chdir(BASE)
COSYVOICE_DIR = f'{BASE}/CosyVoice'
if not os.path.exists(f'{COSYVOICE_DIR}/.git'):
    !rm -rf {COSYVOICE_DIR}
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSYVOICE_DIR}
    !cd {COSYVOICE_DIR} && git submodule update --init --recursive
print("Repo cloned.")

# Install CosyVoice deps — skip torch/torchaudio (already installed)
# Install line by line to avoid one bad package killing everything
!cd {COSYVOICE_DIR} && cat requirements.txt | grep -v '^torch' | grep -v tensorrt | grep -v ttsfrd | while read pkg; do pip install -q "$pkg" 2>/dev/null; done

# Ensure critical packages are installed
!pip install -q hyperpyyaml modelscope onnxruntime soundfile librosa \
    openai-whisper conformer diffsptk inflect pydub einops omegaconf \
    huggingface_hub datasets torchaudio num2words 'transformers>=4.45,<4.50' \
    2>&1 | tail -3

print("\nInstall complete!")

## Step 2: HuggingFace Login + Download Model + Data

In [ ]:
# --- HuggingFace Login ---
try:
    if IS_COLAB:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

if not os.environ.get('HF_TOKEN'):
    print("HF_TOKEN not set.")
    print("Get a token at: https://huggingface.co/settings/tokens")
    print("Make sure 'Access to public gated repos' is enabled.")
    from huggingface_hub import login
    login()
else:
    from huggingface_hub import HfApi
    try:
        print(f"HF user: {HfApi().whoami()['name']}")
    except Exception:
        from huggingface_hub import login
        login()

In [ ]:
# --- Download CosyVoice2 Model ---
sys.path.insert(0, COSYVOICE_DIR)
sys.path.insert(0, f'{COSYVOICE_DIR}/third_party/Matcha-TTS')

MODEL_DIR = f'{COSYVOICE_DIR}/pretrained_models/CosyVoice2-0.5B'
if not os.path.exists(f'{MODEL_DIR}/llm.pt'):
    from huggingface_hub import snapshot_download
    snapshot_download('FunAudioLLM/CosyVoice2-0.5B', local_dir=MODEL_DIR)
    print("Model downloaded!")
else:
    print("Model already downloaded.")

In [ ]:
# --- Download Svarah Indian English Reference Voices ---
import io, random
import numpy as np
import soundfile as sf

DATA_DIR = f'{BASE}/data'
BACKUP_DIR = f'{BASE}/backup_cosyvoice2'
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = '/content/drive/MyDrive/indian_tts_cosyvoice2'
os.makedirs(BACKUP_DIR, exist_ok=True)
os.makedirs(f'{DATA_DIR}/svarah/male', exist_ok=True)
os.makedirs(f'{DATA_DIR}/svarah/female', exist_ok=True)

import glob
existing_male = glob.glob(f'{DATA_DIR}/svarah/male/svarah_*.wav')
existing_female = glob.glob(f'{DATA_DIR}/svarah/female/svarah_*.wav')

if len(existing_male) >= 5 and len(existing_female) >= 5:
    print(f"Svarah data exists: {len(existing_male)} male, {len(existing_female)} female clips")
else:
    print("Downloading Svarah reference voices...")
    from datasets import load_dataset, Audio

    ds = load_dataset("ai4bharat/Svarah", split="test")
    ds = ds.cast_column("audio_filepath", Audio(decode=False))

    male_saved = 0
    female_saved = 0
    manifests = []

    for i, sample in enumerate(ds):
        if male_saved >= 20 and female_saved >= 20:
            break

        gender = (sample.get('gender') or '').strip().lower()
        text = (sample.get('text') or '').strip()
        if gender not in ('male', 'female') or len(text) < 3:
            continue
        if gender == 'male' and male_saved >= 20:
            continue
        if gender == 'female' and female_saved >= 20:
            continue

        audio_data = sample.get('audio_filepath')
        if not audio_data or not audio_data.get('bytes'):
            continue

        try:
            arr, sr = sf.read(io.BytesIO(audio_data['bytes']))
            arr = arr.astype(np.float32)
            if arr.ndim > 1:
                arr = arr.mean(axis=1)
        except Exception:
            continue

        duration = len(arr) / sr
        if duration < 3.0 or duration > 15.0:
            continue

        mx = np.abs(arr).max()
        if mx > 0:
            arr = arr / mx * 0.95

        filepath = f'{DATA_DIR}/svarah/{gender}/svarah_{i:06d}.wav'
        sf.write(filepath, arr, sr)
        manifests.append(f"{filepath}|{0 if gender=='male' else 1}|{text}")

        if gender == 'male':
            male_saved += 1
        else:
            female_saved += 1

    # Write manifest
    with open(f'{DATA_DIR}/train.txt', 'w') as f:
        f.write('# audio_path|speaker_id|text\n')
        for line in manifests:
            f.write(line + '\n')

    print(f"Done! Male: {male_saved} | Female: {female_saved} clips")

print("\nStep 2 complete!")

## Step 3: Load CosyVoice 2 Model

In [ ]:
os.chdir(COSYVOICE_DIR)
from cosyvoice.cli.cosyvoice import CosyVoice2
import torchaudio

cosyvoice = CosyVoice2('pretrained_models/CosyVoice2-0.5B')
print(f"Model loaded! Sample rate: {cosyvoice.sample_rate}")

## Step 4: Browse & Pick Reference Voices

Listen to several clips and pick the ones with the accent you want.
- For **lighter Indian accent** (corporate English): pick speakers who sound less regional
- For **stronger Indian accent**: pick speakers with more regional flavor

The model clones whatever accent it hears in the reference.

In [ ]:
import IPython.display as ipd

male_wavs = sorted(glob.glob(f'{DATA_DIR}/svarah/male/svarah_*.wav'))
female_wavs = sorted(glob.glob(f'{DATA_DIR}/svarah/female/svarah_*.wav'))

# Read manifest for transcripts
transcripts = {}
manifest_path = f'{DATA_DIR}/train.txt'
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('|')
            transcripts[parts[0]] = parts[2]

print("=== MALE SPEAKERS — pick the best accent ===")
for i, wav in enumerate(male_wavs[:10]):
    data, sr = sf.read(wav)
    dur = len(data) / sr
    txt = transcripts.get(wav, '(no transcript)')
    print(f"\nMale #{i} ({dur:.1f}s): {txt[:70]}...")
    ipd.display(ipd.Audio(wav))

print("\n\n=== FEMALE SPEAKERS — pick the best accent ===")
for i, wav in enumerate(female_wavs[:10]):
    data, sr = sf.read(wav)
    dur = len(data) / sr
    txt = transcripts.get(wav, '(no transcript)')
    print(f"\nFemale #{i} ({dur:.1f}s): {txt[:70]}...")
    ipd.display(ipd.Audio(wav))

In [ ]:
# === SET YOUR CHOSEN REFERENCE CLIPS HERE ===
# Change the index numbers based on what sounded best above
MALE_IDX = 0      # Change this to the male clip # you liked
FEMALE_IDX = 0     # Change this to the female clip # you liked

male_ref = male_wavs[MALE_IDX]
female_ref = female_wavs[FEMALE_IDX]
male_ref_text = transcripts.get(male_ref, '')
female_ref_text = transcripts.get(female_ref, '')

print(f"Male ref: #{MALE_IDX} — {male_ref_text[:60]}...")
print(f"Female ref: #{FEMALE_IDX} — {female_ref_text[:60]}...")

## Step 5: Generate Podcast (Zero-Shot — NO training)

In [ ]:
import time

PODCAST_SCRIPT = [
    ("female", "Welcome to AI India, the podcast where we explore how artificial intelligence is transforming our country. I am Priya."),
    ("male", "And I am Arjun. Today we are talking about something really exciting. The rise of Indian AI startups."),
    ("female", "That is right, Arjun. India now has over three hundred AI startups, and that number is growing every single month."),
    ("male", "What I find really interesting is that many of these companies are solving uniquely Indian problems. Like agriculture, healthcare in rural areas, and education."),
    ("female", "Absolutely. Take for example an AI system that can detect crop diseases just by looking at a photo taken on a farmer's mobile phone."),
    ("male", "And in healthcare, AI models are now screening for conditions like diabetic retinopathy and tuberculosis in areas where there are very few doctors available."),
    ("female", "The language barrier is another big challenge that AI is helping with. India has twenty two official languages and hundreds of dialects."),
    ("male", "Exactly. And that is precisely why building speech technology like text to speech systems in Indian languages is so important."),
    ("female", "Speaking of which, the progress in Indian language AI has been remarkable. Models can now understand and generate speech in Hindi, Tamil, Bengali, and many more."),
    ("male", "The government has also been supportive with initiatives to build open source datasets for Indian languages. This is a game changer."),
    ("female", "So what do you think is next for AI in India, Arjun?"),
    ("male", "I believe we will see AI becoming a part of everyday life. From voice assistants that truly understand Indian accents, to AI tutors that teach children in their mother tongue."),
    ("female", "That is a beautiful vision. And it all starts with building the right foundation, the right data, the right models, and the right talent."),
    ("male", "Could not agree more. India has the talent, and now we are building the tools."),
    ("female", "That is all for today's episode of AI India. Thank you for listening, and we will see you next week."),
    ("male", "Goodbye everyone, and keep innovating!"),
]

OUTPUT_DIR = f'{BASE}/outputs/cosyvoice2_podcast'
os.makedirs(OUTPUT_DIR, exist_ok=True)
sr = cosyvoice.sample_rate

all_segments = []
silence = np.zeros(int(sr * 0.6))

print(f"Generating podcast ({len(PODCAST_SCRIPT)} lines)...\n")
start = time.time()

for i, (speaker, text) in enumerate(PODCAST_SCRIPT):
    name = "Priya" if speaker == "female" else "Arjun"
    ref_wav = female_ref if speaker == "female" else male_ref
    ref_txt = female_ref_text if speaker == "female" else male_ref_text

    gen_start = time.time()
    chunks = []
    for result in cosyvoice.inference_zero_shot(text, ref_txt, ref_wav, stream=False):
        chunks.append(result['tts_speech'].squeeze().numpy())

    audio = np.concatenate(chunks) if chunks else np.zeros(sr)
    gen_time = time.time() - gen_start
    duration = len(audio) / sr

    print(f"  [{name:5s}] {duration:.1f}s (gen: {gen_time:.1f}s) | {text[:50]}...")
    sf.write(f'{OUTPUT_DIR}/line_{i:02d}_{speaker}.wav', audio, sr)

    if i > 0:
        all_segments.append(silence)
    all_segments.append(audio)

# Combine
full_audio = np.concatenate(all_segments)
PODCAST_PATH = f'{OUTPUT_DIR}/podcast_full.wav'
sf.write(PODCAST_PATH, full_audio, sr)

total_time = time.time() - start
total_dur = len(full_audio) / sr
print(f"\nPodcast done!")
print(f"  Duration: {total_dur:.0f}s ({total_dur/60:.1f} min)")
print(f"  Generation time: {total_time:.0f}s")
print(f"  RTF: {total_time/total_dur:.2f}x")

## Step 6: Listen & Save

In [ ]:
print("=" * 60)
print("  CosyVoice 2 — Zero-Shot Indian English Podcast")
print("=" * 60)

print("\nFull podcast:")
ipd.display(ipd.Audio(PODCAST_PATH))

print("\nIndividual lines:")
for i, (speaker, text) in enumerate(PODCAST_SCRIPT[:6]):
    name = "Priya" if speaker == "female" else "Arjun"
    wav_path = f'{OUTPUT_DIR}/line_{i:02d}_{speaker}.wav'
    print(f"\n  [{name}] {text[:60]}...")
    ipd.display(ipd.Audio(wav_path))

In [ ]:
# Save to backup
import shutil
shutil.copy2(PODCAST_PATH, os.path.join(BACKUP_DIR, 'podcast_zero_shot.wav'))
for f in glob.glob(f'{OUTPUT_DIR}/line_*.wav'):
    shutil.copy2(f, BACKUP_DIR)
print(f"All files saved to: {BACKUP_DIR}")
print(f"\nDownload podcast_full.wav from: {PODCAST_PATH}")

## Try Different Voices

If the accent wasn't right, go back to Step 4, change `MALE_IDX` and `FEMALE_IDX`, and re-run Steps 4-6.

In [ ]:
# Quick single-sentence test with a specific voice
test_text = "Good morning everyone. Today we will discuss the quarterly results."

for result in cosyvoice.inference_zero_shot(
    test_text, male_ref_text, male_ref, stream=False
):
    test_audio = result['tts_speech'].squeeze().numpy()

print("[MALE]")
ipd.display(ipd.Audio(test_audio, rate=sr))

for result in cosyvoice.inference_zero_shot(
    test_text, female_ref_text, female_ref, stream=False
):
    test_audio = result['tts_speech'].squeeze().numpy()

print("[FEMALE]")
ipd.display(ipd.Audio(test_audio, rate=sr))